In [39]:
!pip install requests beautifulsoup4 lxml pandas pypdf python-dateutil

In [40]:
import os
import re
import time
import hashlib
import mimetypes
from collections import deque
from datetime import datetime
from urllib.parse import (
    urlparse,
    urljoin,
    urldefrag,
    parse_qsl,
    urlencode
)

import requests
import pandas as pd

from bs4 import BeautifulSoup
from pypdf import PdfReader

In [41]:
# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = "portfolio_data"

MAX_INTERNAL_PAGES = 100
REQUEST_TIMEOUT = 20
REQUEST_DELAY = 0.7

# Maximum amount of text retained from one page.
# We preserve enough evidence without creating enormous CSV cells.
MAX_TEXT_LENGTH = 100000

# Supported document extensions
DOCUMENT_EXTENSIONS = {
    ".pdf",
    ".doc",
    ".docx",
    ".txt",
    ".csv",
    ".xls",
    ".xlsx",
    ".ppt",
    ".pptx"
}

os.makedirs(DATA_DIR, exist_ok=True)

print("Data directory:", os.path.abspath(DATA_DIR))


Data directory: /Users/varshh06/Desktop/1. PORTFOLIO INTELLIGENCE/portfolio_data


In [42]:
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "PortfolioIntelligenceCareerAnalyzer/1.0 "
        "(educational research project)"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9"
})

In [43]:
def normalize_url(url):
    """
    Normalize a URL so duplicate URLs don't get crawled repeatedly.
    """

    if not url:
        return None

    url = str(url).strip()

    # Remove fragment
    url = urldefrag(url)[0]

    parsed = urlparse(url)

    if parsed.scheme.lower() not in ("http", "https"):
        return None

    scheme = parsed.scheme.lower()

    # Host
    domain = parsed.netloc.lower()

    # Remove default ports
    if domain.endswith(":80"):
        domain = domain[:-3]

    if domain.endswith(":443"):
        domain = domain[:-4]

    # Normalize path
    path = parsed.path or "/"

    path = re.sub(
        r"/+",
        "/",
        path
    )

    if path != "/" and path.endswith("/"):
        path = path.rstrip("/")

    # Keep useful query parameters only
    query = parsed.query

    if query:

        useful_params = []

        for key, value in parse_qsl(
            query,
            keep_blank_values=True
        ):

            key_lower = key.lower()

            if key_lower.startswith("utm_"):
                continue

            if key_lower in {
                "fbclid",
                "gclid",
                "ref",
                "source"
            }:
                continue

            useful_params.append(
                (key, value)
            )

        query = urlencode(
            useful_params
        )

    result = f"{scheme}://{domain}{path}"

    if query:
        result += f"?{query}"

    return result

In [44]:
portfolio_url = input(
    "Enter candidate portfolio URL: "
).strip()

portfolio_url = normalize_url(
    portfolio_url
)

if not portfolio_url:
    raise ValueError(
        "Invalid portfolio URL."
    )

portfolio_domain = urlparse(
    portfolio_url
).netloc.lower()

print()
print("Portfolio URL:")
print(portfolio_url)

print()
print("Portfolio domain:")
print(portfolio_domain)


Portfolio URL:
https://varshh-hub.github.io/VARSHA---portfolio

Portfolio domain:
varshh-hub.github.io


In [45]:
def get_next_candidate_id():
    """
    Generates:
        CAND_0001
        CAND_0002
        CAND_0003
        ...

    Existing candidates are never overwritten.
    """

    path = os.path.join(
        DATA_DIR,
        "candidates.csv"
    )

    if not os.path.exists(path):
        return "CAND_0001"

    try:
        df = pd.read_csv(path)

    except Exception:
        return "CAND_0001"

    if df.empty or "candidate_id" not in df.columns:
        return "CAND_0001"

    numbers = (
        df["candidate_id"]
        .astype(str)
        .str.extract(
            r"CAND_(\d+)",
            expand=False
        )
    )

    numbers = pd.to_numeric(
        numbers,
        errors="coerce"
    ).dropna()

    if numbers.empty:
        next_number = 1
    else:
        next_number = int(
            numbers.max()
        ) + 1

    return f"CAND_{next_number:04d}"

In [46]:
candidate_id = get_next_candidate_id()

print(
    "New candidate ID:",
    candidate_id
)

New candidate ID: CAND_0002


In [47]:
def is_internal_url(url):
    """
    True only when the URL belongs to the portfolio domain.
    """

    if not url:
        return False

    try:
        domain = urlparse(
            url
        ).netloc.lower()

        return (
            domain == portfolio_domain
            or domain.endswith(
                "." + portfolio_domain
            )
        )

    except Exception:
        return False

In [48]:
def get_extension(url):

    path = urlparse(
        url
    ).path.lower()

    return os.path.splitext(
        path
    )[1]


def is_document_url(url):

    return (
        get_extension(url)
        in DOCUMENT_EXTENSIONS
    )

In [49]:
PLATFORMS = {
    "github.com": "GitHub",
    "gitlab.com": "GitLab",
    "bitbucket.org": "Bitbucket",

    "linkedin.com": "LinkedIn",

    "kaggle.com": "Kaggle",

    "leetcode.com": "LeetCode",
    "hackerrank.com": "HackerRank",
    "codeforces.com": "Codeforces",
    "codechef.com": "CodeChef",

    "geeksforgeeks.org": "GeeksforGeeks",

    "medium.com": "Medium",
    "dev.to": "Dev.to",

    "stackoverflow.com": "Stack Overflow",

    "credly.com": "Credly",

    "coursera.org": "Coursera",
    "udemy.com": "Udemy",

    "pypi.org": "PyPI",
    "npmjs.com": "NPM"
}


def detect_platform(url):

    if not url:
        return "Unknown"

    lower = url.lower()

    if lower.startswith("mailto:"):
        return "Email"

    if lower.startswith("tel:"):
        return "Phone"

    if is_document_url(url):
        return "Document"

    try:
        domain = urlparse(
            url
        ).netloc.lower()

    except Exception:
        return "Unknown"

    # Remove www.
    domain = domain.removeprefix(
        "www."
    )

    for website, platform in PLATFORMS.items():

        if (
            domain == website
            or domain.endswith(
                "." + website
            )
        ):
            return platform

    return "Other"

In [50]:
def download_url(url):

    try:

        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True
        )

        return {
            "requested_url": url,
            "final_url": response.url,
            "status_code": response.status_code,
            "content_type": response.headers.get(
                "Content-Type",
                ""
            ),
            "content": response.content,
            "error": None
        }

    except Exception as e:

        return {
            "requested_url": url,
            "final_url": None,
            "status_code": None,
            "content_type": "",
            "content": None,
            "error": str(e)
        }

In [51]:
def extract_links(
    soup,
    current_url
):

    internal_links = set()
    external_links = set()
    document_links = set()
    email_links = set()
    phone_links = set()

    for tag in soup.find_all(
        "a",
        href=True
    ):

        href = tag.get(
            "href",
            ""
        ).strip()

        if not href:
            continue

        lower_href = href.lower()

        # Ignore JavaScript/data links
        if lower_href.startswith(
            (
                "javascript:",
                "#",
                "data:"
            )
        ):
            continue

        # Email
        if lower_href.startswith(
            "mailto:"
        ):

            email_links.add(
                href
            )

            continue

        # Phone
        if lower_href.startswith(
            "tel:"
        ):

            phone_links.add(
                href
            )

            continue

        absolute_url = urljoin(
            current_url,
            href
        )

        normalized = normalize_url(
            absolute_url
        )

        if not normalized:
            continue

        # -----------------------------------------
        # DOCUMENT
        # -----------------------------------------

        if is_document_url(
            normalized
        ):

            document_links.add(
                normalized
            )

            continue

        # -----------------------------------------
        # INTERNAL HTML
        # -----------------------------------------

        if is_internal_url(
            normalized
        ):

            internal_links.add(
                normalized
            )

        # -----------------------------------------
        # EXTERNAL
        # -----------------------------------------

        else:

            external_links.add(
                normalized
            )

    return {
        "internal": sorted(
            internal_links
        ),
        "external": sorted(
            external_links
        ),
        "documents": sorted(
            document_links
        ),
        "emails": sorted(
            email_links
        ),
        "phones": sorted(
            phone_links
        )
    }

In [52]:
def clean_text(text):

    if not text:
        return ""

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [53]:
def parse_html(
    html,
    current_url
):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    # -----------------------------------------
    # TITLE
    # -----------------------------------------

    title = ""

    if soup.title:

        title = clean_text(
            soup.title.get_text(
                " ",
                strip=True
            )
        )

    # -----------------------------------------
    # META DESCRIPTION
    # -----------------------------------------

    meta_description = ""

    meta = soup.find(
        "meta",
        attrs={
            "name": re.compile(
                "^description$",
                re.I
            )
        }
    )

    if meta:

        meta_description = clean_text(
            meta.get(
                "content",
                ""
            )
        )

    # -----------------------------------------
    # REMOVE NON-CONTENT
    # -----------------------------------------

    for tag in soup.find_all([
        "script",
        "style",
        "noscript",
        "template"
    ]):

        tag.decompose()

    # -----------------------------------------
    # HEADINGS
    # -----------------------------------------

    headings = []

    for tag in soup.find_all([
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6"
    ]):

        text = clean_text(
            tag.get_text(
                " ",
                strip=True
            )
        )

        if text:

            headings.append({
                "tag": tag.name,
                "text": text
            })

    # -----------------------------------------
    # PARAGRAPHS
    # -----------------------------------------

    paragraphs = []

    for tag in soup.find_all("p"):

        text = clean_text(
            tag.get_text(
                " ",
                strip=True
            )
        )

        if text:
            paragraphs.append(
                text
            )

    # -----------------------------------------
    # LIST ITEMS
    # -----------------------------------------

    list_items = []

    for li in soup.find_all("li"):

        text = clean_text(
            li.get_text(
                " ",
                strip=True
            )
        )

        if text:
            list_items.append(
                text
            )

    # -----------------------------------------
    # IMAGES
    # -----------------------------------------

    images = []

    for img in soup.find_all("img"):

        src = (
            img.get("src")
            or img.get("data-src")
            or img.get("data-lazy-src")
        )

        if not src:
            continue

        images.append({

            "url":
                urljoin(
                    current_url,
                    src
                ),

            "alt":
                clean_text(
                    img.get(
                        "alt",
                        ""
                    )
                ),

            "title":
                clean_text(
                    img.get(
                        "title",
                        ""
                    )
                )
        })

    # -----------------------------------------
    # LINKS
    # -----------------------------------------

    links = extract_links(
        soup,
        current_url
    )

    # -----------------------------------------
    # MAIN TEXT
    # -----------------------------------------

    text = soup.get_text(
        separator="\n",
        strip=True
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    text = text[:MAX_TEXT_LENGTH]

    return {

        "title":
            title,

        "meta_description":
            meta_description,

        "headings":
            headings,

        "paragraphs":
            paragraphs,

        "list_items":
            list_items,

        "images":
            images,

        "text":
            text,

        "links":
            links
    }

In [54]:
PAGE_TYPE_KEYWORDS = {

    "projects": [
        "project",
        "projects",
        "portfolio",
        "work",
        "case study",
        "case studies"
    ],

    "experience": [
        "experience",
        "employment",
        "work experience",
        "professional experience"
    ],

    "internship": [
        "internship",
        "intern",
        "internships",
        "training"
    ],

    "education": [
        "education",
        "academic",
        "degree",
        "qualification",
        "university",
        "college"
    ],

    "certifications": [
        "certificate",
        "certification",
        "certifications",
        "credential",
        "credentials"
    ],

    "skills": [
        "skill",
        "skills",
        "technologies",
        "technology",
        "technical skills",
        "tools"
    ],

    "about": [
        "about",
        "about me",
        "profile",
        "who i am",
        "summary"
    ],

    "contact": [
        "contact",
        "email",
        "phone",
        "reach me"
    ]
}

In [55]:
def classify_page(
    url,
    title,
    headings,
    text
):

    combined = " ".join([

        url or "",

        title or "",

        " ".join(
            h.get("text", "")
            for h in headings
        ),

        text[:5000] if text else ""

    ]).lower()

    scores = {}

    for page_type, keywords in (
        PAGE_TYPE_KEYWORDS.items()
    ):

        score = 0

        for keyword in keywords:

            if keyword.lower() in combined:

                score += 1

        scores[page_type] = score

    best_type = max(
        scores,
        key=scores.get
    )

    if scores[best_type] == 0:

        return "other"

    return best_type

In [56]:
SKILL_ALIASES = {

    # Programming
    "python": [
        "python"
    ],

    "java": [
        "java"
    ],

    "javascript": [
        "javascript",
        "js"
    ],

    "typescript": [
        "typescript"
    ],

    "c++": [
        "c++",
        "cpp"
    ],

    "c": [
        "c programming"
    ],

    "c#": [
        "c#",
        "c sharp"
    ],

    # Data
    "sql": [
        "sql",
        "structured query language"
    ],

    "excel": [
        "microsoft excel",
        "ms excel",
        "excel"
    ],

    "pandas": [
        "pandas"
    ],

    "numpy": [
        "numpy"
    ],

    "scikit-learn": [
        "scikit-learn",
        "sklearn",
        "scikit learn"
    ],

    "tensorflow": [
        "tensorflow"
    ],

    "pytorch": [
        "pytorch"
    ],

    "xgboost": [
        "xgboost"
    ],

    "machine learning": [
        "machine learning",
        "ml"
    ],

    "deep learning": [
        "deep learning"
    ],

    "statistics": [
        "statistics",
        "statistical analysis"
    ],

    # BI
    "power bi": [
        "power bi",
        "powerbi"
    ],

    "tableau": [
        "tableau"
    ],

    # Web
    "html": [
        "html"
    ],

    "css": [
        "css"
    ],

    "bootstrap": [
        "bootstrap"
    ],

    "react": [
        "react",
        "react.js",
        "reactjs"
    ],

    "node.js": [
        "node.js",
        "nodejs"
    ],

    # Databases
    "mysql": [
        "mysql"
    ],

    "postgresql": [
        "postgresql",
        "postgres"
    ],

    "mongodb": [
        "mongodb",
        "mongo db"
    ],

    # Cloud / DevOps
    "aws": [
        "aws",
        "amazon web services"
    ],

    "azure": [
        "azure",
        "microsoft azure"
    ],

    "gcp": [
        "gcp",
        "google cloud",
        "google cloud platform"
    ],

    "docker": [
        "docker"
    ],

    "git": [
        "git"
    ],

    "github": [
        "github"
    ],

    # Data tools
    "jupyter": [
        "jupyter",
        "jupyter notebook",
        "jupyter notebooks"
    ],

    "powerpoint": [
        "powerpoint",
        "microsoft powerpoint"
    ]
}

In [57]:
def extract_skills(text):

    if not text:
        return []

    text_lower = text.lower()

    found = []

    for canonical_skill, aliases in (
        SKILL_ALIASES.items()
    ):

        for alias in aliases:

            # Word-boundary-ish matching.
            # Special characters are escaped.
            pattern = (
                r"(?<!\w)"
                + re.escape(alias.lower())
                + r"(?!\w)"
            )

            if re.search(
                pattern,
                text_lower
            ):

                found.append(
                    canonical_skill
                )

                break

    return sorted(
        set(found)
    )

In [58]:
def extract_skill_evidence(
    text,
    source_url,
    source_type
):

    records = []

    if not text:
        return records

    # Split into reasonably useful sentences/lines
    chunks = re.split(
        r"(?<=[.!?])\s+|\n+",
        text
    )

    for chunk in chunks:

        chunk = clean_text(
            chunk
        )

        if not chunk:
            continue

        skills = extract_skills(
            chunk
        )

        for skill in skills:

            records.append({

                "skill":
                    skill,

                "evidence_text":
                    chunk,

                "source_url":
                    source_url,

                "source_type":
                    source_type
            })

    return records

In [59]:
def extract_projects_from_page(
    page
):

    records = []

    page_type = page.get(
        "page_type",
        ""
    )

    if page_type not in {
        "projects",
        "other"
    }:
        return records

    text = page.get(
        "text",
        ""
    )

    if not text:
        return records

    soup_text = text

    # Search for common project headings
    project_pattern = re.compile(
        r"(?im)"
        r"(?:^|\n)"
        r"(?:project|projects|case study|case studies)"
        r"\s*[:\-]?\s*(.+)"
    )

    matches = project_pattern.findall(
        soup_text
    )

    if matches:

        for match in matches:

            project_name = clean_text(
                match
            )

            if project_name:

                records.append({

                    "project_name":
                        project_name,

                    "description":
                        "",

                    "skills":
                        ", ".join(
                            extract_skills(
                                project_name
                            )
                        ),

                    "source_url":
                        page["url"],

                    "evidence_text":
                        project_name,

                    "source_type":
                        "project"
                })

    # Also inspect headings
    for heading in page.get(
        "headings",
        []
    ):

        heading_text = clean_text(
            heading.get(
                "text",
                ""
            )
        )

        if not heading_text:
            continue

        lower = heading_text.lower()

        # Skip generic section headings
        if lower in {
            "projects",
            "my projects",
            "portfolio",
            "work"
        }:
            continue

        skills = extract_skills(
            heading_text
        )

        # Project-like headings often contain
        # technology names or descriptive titles.
        if skills or len(
            heading_text.split()
        ) >= 2:

            records.append({

                "project_name":
                    heading_text,

                "description":
                    "",

                "skills":
                    ", ".join(
                        skills
                    ),

                "source_url":
                    page["url"],

                "evidence_text":
                    heading_text,

                "source_type":
                    "project"
            })

    return records

In [60]:
def extract_experience_from_page(
    page
):

    records = []

    page_type = page.get(
        "page_type",
        ""
    )

    if page_type not in {
        "experience",
        "internship"
    }:
        return records

    text = page.get(
        "text",
        ""
    )

    if not text:
        return records

    source_type = (
        "internship"
        if page_type == "internship"
        else "experience"
    )

    # Find lines containing common experience indicators
    lines = [
        clean_text(line)
        for line in text.splitlines()
        if clean_text(line)
    ]

    for line in lines:

        lower = line.lower()

        if any(
            keyword in lower
            for keyword in [
                "intern",
                "developer",
                "engineer",
                "analyst",
                "scientist",
                "manager",
                "designer",
                "consultant",
                "trainee",
                "associate"
            ]
        ):

            records.append({

                "role":
                    line,

                "company":
                    "",

                "description":
                    line,

                "skills":
                    ", ".join(
                        extract_skills(
                            line
                        )
                    ),

                "source_url":
                    page["url"],

                "evidence_text":
                    line,

                "source_type":
                    source_type
            })

    return records

In [61]:
def extract_education_from_page(page):

    records = []

    text = page.get(
        "text",
        ""
    )

    if not text:
        return records

    # Normalize line breaks
    lines = [
        clean_text(line)
        for line in text.splitlines()
        if clean_text(line)
    ]

    education_keywords = [
        "education",
        "academic",
        "school",
        "college",
        "university",

        "bachelor",
        "bachelors",
        "b.tech",
        "btech",
        "b.e",
        "b.sc",
        "bsc",
        "bca",

        "master",
        "masters",
        "m.tech",
        "mtech",
        "m.e",
        "m.sc",
        "msc",
        "mca",

        "degree",
        "diploma",

        "higher secondary",
        "secondary school",
        "high school",

        "10th",
        "12th",

        "pursuing",
        "currently studying",
        "currently pursuing",
        "ongoing",

        "undergraduate",
        "postgraduate"
    ]

    # --------------------------------------------------
    # Find education-related lines
    # --------------------------------------------------

    matched_indices = []

    for i, line in enumerate(lines):

        lower = line.lower()

        if any(
            keyword in lower
            for keyword in education_keywords
        ):

            matched_indices.append(i)

    # --------------------------------------------------
    # Build context blocks
    # --------------------------------------------------

    used = set()

    for index in matched_indices:

        if index in used:
            continue

        start = max(
            0,
            index - 2
        )

        end = min(
            len(lines),
            index + 3
        )

        block_lines = lines[
            start:end
        ]

        for i in range(
            start,
            end
        ):
            used.add(i)

        evidence = "\n".join(
            block_lines
        )

        qualification = lines[
            index
        ]

        records.append({

            "qualification":
                qualification,

            "institution":
                "",

            "year":
                "",

            "source_url":
                page["url"],

            "evidence_text":
                evidence,

            "source_type":
                "education",

            "evidence_type":
                "education"
        })

    return records

In [62]:
def extract_certifications_from_page(page):

    records = []

    text = page.get(
        "text",
        ""
    )

    if not text:
        return records

    lines = [
        clean_text(line)
        for line in text.splitlines()
        if clean_text(line)
    ]

    certification_keywords = [
        "certificate",
        "certification",
        "certifications",
        "certified",
        "credential",
        "credentials",
        "course completion",
        "completed course",
        "course certificate"
    ]

    matched_indices = []

    for i, line in enumerate(lines):

        lower = line.lower()

        if any(
            keyword in lower
            for keyword in certification_keywords
        ):

            matched_indices.append(i)

    used = set()

    for index in matched_indices:

        if index in used:
            continue

        start = max(
            0,
            index - 2
        )

        end = min(
            len(lines),
            index + 3
        )

        block_lines = lines[
            start:end
        ]

        for i in range(
            start,
            end
        ):
            used.add(i)

        evidence = "\n".join(
            block_lines
        )

        records.append({

            "certificate_name":
                lines[index],

            "issuer":
                "",

            "date":
                "",

            "source_url":
                page["url"],

            "evidence_text":
                evidence,

            "source_type":
                "certification",

            "evidence_type":
                "certification"
        })

    return records

In [63]:
def crawl_portfolio(
    start_url
):

    start_url = normalize_url(
        start_url
    )

    queue = deque([
        start_url
    ])

    visited_pages = set()

    pages = []

    external_links = set()
    document_links = set()
    email_links = set()
    phone_links = set()

    while queue:

        if len(
            visited_pages
        ) >= MAX_INTERNAL_PAGES:

            print(
                "Maximum page limit reached."
            )

            break

        current_url = queue.popleft()

        if current_url in visited_pages:
            continue

        visited_pages.add(
            current_url
        )

        print(
            f"[{len(visited_pages)}/"
            f"{MAX_INTERNAL_PAGES}] "
            f"{current_url}"
        )

        result = download_url(
            current_url
        )

        # Request failed
        if result["content"] is None:

            pages.append({

                "url":
                    current_url,

                "final_url":
                    result["final_url"],

                "status_code":
                    result["status_code"],

                "content_type":
                    result["content_type"],

                "title":
                    "",

                "meta_description":
                    "",

                "headings":
                    [],

                "paragraphs":
                    [],

                "list_items":
                    [],

                "images":
                    [],

                "text":
                    "",

                "internal_links":
                    [],

                "external_links":
                    [],

                "document_links":
                    [],

                "page_type":
                    "error",

                "error":
                    result["error"]
            })

            continue

        content_type = (
            result["content_type"]
            or ""
        ).lower()

        # Only HTML is processed here
        if "text/html" not in content_type:

            continue

        parsed = parse_html(
            result["content"],
            result["final_url"]
        )

        links = parsed["links"]

        page_type = classify_page(
            current_url,
            parsed["title"],
            parsed["headings"],
            parsed["text"]
        )

        # Collect discovered links
        external_links.update(
            links["external"]
        )

        document_links.update(
            links["documents"]
        )

        email_links.update(
            links["emails"]
        )

        phone_links.update(
            links["phones"]
        )

        page = {

            "url":
                current_url,

            "final_url":
                result["final_url"],

            "status_code":
                result["status_code"],

            "content_type":
                result["content_type"],

            "title":
                parsed["title"],

            "meta_description":
                parsed["meta_description"],

            "headings":
                parsed["headings"],

            "paragraphs":
                parsed["paragraphs"],

            "list_items":
                parsed["list_items"],

            "images":
                parsed["images"],

            "text":
                parsed["text"],

            "internal_links":
                links["internal"],

            "external_links":
                links["external"],

            "document_links":
                links["documents"],

            "page_type":
                page_type,

            "error":
                None
        }

        pages.append(
            page
        )

        # IMPORTANT:
        # Only internal HTML links enter queue.
        for link in links["internal"]:

            if link not in visited_pages:

                queue.append(
                    link
                )

        time.sleep(
            REQUEST_DELAY
        )

    return {

        "pages":
            pages,

        "external_links":
            sorted(
                external_links
            ),

        "document_links":
            sorted(
                document_links
            ),

        "email_links":
            sorted(
                email_links
            ),

        "phone_links":
            sorted(
                phone_links
            )
    }

In [64]:
crawl_result = crawl_portfolio(
    portfolio_url
)

pages = crawl_result["pages"]

print()
print("=" * 70)
print("CRAWLING COMPLETE")
print("=" * 70)

print(
    "Candidate:",
    candidate_id
)

print(
    "Internal HTML pages:",
    len(pages)
)

print(
    "External links:",
    len(
        crawl_result[
            "external_links"
        ]
    )
)

print(
    "Documents:",
    len(
        crawl_result[
            "document_links"
        ]
    )
)

print(
    "Email links:",
    len(
        crawl_result[
            "email_links"
        ]
    )
)

print(
    "Phone links:",
    len(
        crawl_result[
            "phone_links"
        ]
    )
)

[1/100] https://varshh-hub.github.io/VARSHA---portfolio

CRAWLING COMPLETE
Candidate: CAND_0002
Internal HTML pages: 1
External links: 2
Documents: 1
Email links: 1
Phone links: 1


In [65]:
page_records = []

for page in pages:

    page_records.append({

        "candidate_id":
            candidate_id,

        "source_url":
            page["url"],

        "final_url":
            page.get(
                "final_url",
                ""
            ),

        "source_type":
            "portfolio_page",

        "page_type":
            page.get(
                "page_type",
                "other"
            ),

        "title":
            page.get(
                "title",
                ""
            ),

        "meta_description":
            page.get(
                "meta_description",
                ""
            ),

        "evidence_text":
            page.get(
                "text",
                ""
            ),

        "status_code":
            page.get(
                "status_code"
            ),

        "content_type":
            page.get(
                "content_type",
                ""
            ),

        "error":
            page.get(
                "error"
            ),

        "scraped_at":
            datetime.now().isoformat()
    })


pages_df = pd.DataFrame(
    page_records
)

print(
    "Page records:",
    len(pages_df)
)

pages_df[
    [
        "candidate_id",
        "source_url",
        "page_type",
        "title"
    ]
]

Page records: 1


,candidate_id,source_url,page_type,title
0,CAND_0002,https://varshh-hub.github.io/VARSHA---portfolio,skills,VARSHA A | Data Science Portfolio


In [66]:
link_records = []

for page in pages:

    source_url = page["url"]

    # -----------------------------------------
    # INTERNAL LINKS
    # -----------------------------------------

    for link in page.get(
        "internal_links",
        []
    ):

        link_records.append({

            "candidate_id":
                candidate_id,

            "source_url":
                source_url,

            "link_url":
                link,

            "link_type":
                "internal",

            "platform":
                "Portfolio",

            "is_internal":
                True,

            "status":
                "discovered"
        })

    # -----------------------------------------
    # EXTERNAL LINKS
    # -----------------------------------------

    for link in page.get(
        "external_links",
        []
    ):

        link_records.append({

            "candidate_id":
                candidate_id,

            "source_url":
                source_url,

            "link_url":
                link,

            "link_type":
                "external",

            "platform":
                detect_platform(
                    link
                ),

            "is_internal":
                False,

            "status":
                "discovered"
        })

    # -----------------------------------------
    # DOCUMENT LINKS
    # -----------------------------------------

    for link in page.get(
        "document_links",
        []
    ):

        link_records.append({

            "candidate_id":
                candidate_id,

            "source_url":
                source_url,

            "link_url":
                link,

            "link_type":
                "document",

            "platform":
                "Document",

            "is_internal":
                is_internal_url(
                    link
                ),

            "status":
                "discovered"
        })


# Add mailto links
for email in crawl_result[
    "email_links"
]:

    link_records.append({

        "candidate_id":
            candidate_id,

        "source_url":
            portfolio_url,

        "link_url":
            email,

        "link_type":
            "email",

        "platform":
            "Email",

        "is_internal":
            False,

        "status":
            "discovered"
    })


# Add phone links
for phone in crawl_result[
    "phone_links"
]:

    link_records.append({

        "candidate_id":
            candidate_id,

        "source_url":
            portfolio_url,

        "link_url":
            phone,

        "link_type":
            "phone",

        "platform":
            "Phone",

        "is_internal":
            False,

        "status":
            "discovered"
    })


links_df = pd.DataFrame(
    link_records
)

if not links_df.empty:

    links_df = links_df.drop_duplicates()

print(
    "Link records:",
    len(links_df)
)

Link records: 5


In [67]:
skill_records = []

for page in pages:

    source_url = page["url"]

    source_type = page.get(
        "page_type",
        "portfolio_page"
    )

    text = page.get(
        "text",
        ""
    )

    evidence = extract_skill_evidence(
        text,
        source_url,
        source_type
    )

    for record in evidence:

        record[
            "candidate_id"
        ] = candidate_id

        record[
            "scraped_at"
        ] = datetime.now().isoformat()

        skill_records.append(
            record
        )


skills_df = pd.DataFrame(
    skill_records
)

if not skills_df.empty:

    skills_df = skills_df[
        [
            "candidate_id",
            "skill",
            "evidence_text",
            "source_url",
            "source_type",
            "scraped_at"
        ]
    ].drop_duplicates()

print(
    "Skill evidence records:",
    len(skills_df)
)

Skill evidence records: 80


In [68]:
project_records = []

for page in pages:

    records = extract_projects_from_page(
        page
    )

    for record in records:

        record[
            "candidate_id"
        ] = candidate_id

        record[
            "scraped_at"
        ] = datetime.now().isoformat()

        project_records.append(
            record
        )


projects_df = pd.DataFrame(
    project_records
)

if not projects_df.empty:

    projects_df = projects_df[
        [
            "candidate_id",
            "project_name",
            "description",
            "skills",
            "source_url",
            "evidence_text",
            "source_type",
            "scraped_at"
        ]
    ].drop_duplicates()

print(
    "Project records:",
    len(projects_df)
)

Project records: 0


In [69]:
experience_records = []

for page in pages:

    records = extract_experience_from_page(
        page
    )

    for record in records:

        record[
            "candidate_id"
        ] = candidate_id

        record[
            "scraped_at"
        ] = datetime.now().isoformat()

        experience_records.append(
            record
        )


experience_df = pd.DataFrame(
    experience_records
)

if not experience_df.empty:

    experience_df = experience_df[
        [
            "candidate_id",
            "role",
            "company",
            "description",
            "skills",
            "source_url",
            "evidence_text",
            "source_type",
            "scraped_at"
        ]
    ].drop_duplicates()

print(
    "Experience records:",
    len(experience_df)
)

Experience records: 0


In [70]:
education_records = []

for page in pages:

    records = extract_education_from_page(
        page
    )

    for record in records:

        record[
            "candidate_id"
        ] = candidate_id

        record[
            "scraped_at"
        ] = datetime.now().isoformat()

        education_records.append(
            record
        )


education_df = pd.DataFrame(
    education_records
)

if not education_df.empty:

    education_df = education_df[
        [
            "candidate_id",
            "qualification",
            "institution",
            "year",
            "source_url",
            "evidence_text",
            "source_type",
            "scraped_at"
        ]
    ].drop_duplicates()

print(
    "Education records:",
    len(education_df)
)

Education records: 10


In [71]:
certification_records = []

for page in pages:

    records = extract_certifications_from_page(
        page
    )

    for record in records:

        record[
            "candidate_id"
        ] = candidate_id

        record[
            "scraped_at"
        ] = datetime.now().isoformat()

        certification_records.append(
            record
        )


certifications_df = pd.DataFrame(
    certification_records
)

if not certifications_df.empty:

    certifications_df = certifications_df[
        [
            "candidate_id",
            "certificate_name",
            "issuer",
            "date",
            "source_url",
            "evidence_text",
            "source_type",
            "scraped_at"
        ]
    ].drop_duplicates()

print(
    "Certification records:",
    len(certifications_df)
)

Certification records: 5


In [72]:
def inspect_document(url):

    result = download_url(
        url
    )

    if result["content"] is None:

        return {

            "url":
                url,

            "document_type":
                get_extension(url),

            "status_code":
                result["status_code"],

            "content_type":
                result["content_type"],

            "final_url":
                result["final_url"],

            "accessible":
                False,

            "is_real_pdf":
                False,

            "error":
                result["error"]
        }

    content = result["content"]

    is_real_pdf = content.startswith(
        b"%PDF"
    )

    return {

        "url":
            url,

        "document_type":
            get_extension(url),

        "status_code":
            result["status_code"],

        "content_type":
            result["content_type"],

        "final_url":
            result["final_url"],

        "accessible":
            bool(
                result["status_code"]
                and 200 <= result[
                    "status_code"
                ] < 400
            ),

        "is_real_pdf":
            is_real_pdf,

        "error":
            None
    }

In [73]:
document_records = []

for document_url in crawl_result[
    "document_links"
]:

    record = inspect_document(
        document_url
    )

    record[
        "candidate_id"
    ] = candidate_id

    record[
        "scraped_at"
    ] = datetime.now().isoformat()

    document_records.append(
        record
    )


documents_df = pd.DataFrame(
    document_records
)

print(
    "Documents:",
    len(documents_df)
)

documents_df

Documents: 1


,url,document_type,status_code,content_type,final_url,accessible,is_real_pdf,error,candidate_id,scraped_at
0,https://varshh-hub.github.io/VARSHA---portfoli...,.pdf,404,text/html; charset=utf-8,https://varshh-hub.github.io/VARSHA---portfoli...,False,False,None,CAND_0002,2026-08-21T13:38:43.099754


In [74]:
candidate_df = pd.DataFrame([{

    "candidate_id":
        candidate_id,

    "portfolio_url":
        portfolio_url,

    "portfolio_domain":
        portfolio_domain,

    "pages_found":
        len(pages),

    "external_links_found":
        len(
            crawl_result[
                "external_links"
            ]
        ),

    "documents_found":
        len(
            crawl_result[
                "document_links"
            ]
        ),

    "emails_found":
        len(
            crawl_result[
                "email_links"
            ]
        ),

    "phones_found":
        len(
            crawl_result[
                "phone_links"
            ]
        ),

    "skills_found":
        (
            skills_df["skill"]
            .nunique()
            if not skills_df.empty
            else 0
        ),

    "projects_found":
        (
            len(projects_df)
            if not projects_df.empty
            else 0
        ),

    "experience_records":
        (
            len(experience_df)
            if not experience_df.empty
            else 0
        ),

    "education_records":
        (
            len(education_df)
            if not education_df.empty
            else 0
        ),

    "certifications_found":
        (
            len(certifications_df)
            if not certifications_df.empty
            else 0
        ),

    "scraped_at":
        datetime.now().isoformat()

}])

candidate_df

,candidate_id,portfolio_url,portfolio_domain,pages_found,external_links_found,documents_found,emails_found,phones_found,skills_found,projects_found,experience_records,education_records,certifications_found,scraped_at
0,CAND_0002,https://varshh-hub.github.io/VARSHA---portfolio,varshh-hub.github.io,1,2,1,1,1,16,0,0,10,5,2026-08-21T13:38:43.109128


In [75]:
def append_to_csv(
    df,
    filename
):

    if df is None or df.empty:

        print(
            f"No records to save: {filename}"
        )

        return

    path = os.path.join(
        DATA_DIR,
        filename
    )

    # Ensure consistent column names
    df = df.copy()

    if os.path.exists(path):

        # Read existing file to determine columns
        try:

            existing = pd.read_csv(
                path,
                nrows=0
            )

            existing_columns = list(
                existing.columns
            )

            # Add missing columns
            for column in existing_columns:

                if column not in df.columns:

                    df[column] = None

            # Add new columns if needed
            for column in df.columns:

                if column not in existing_columns:

                    existing_columns.append(
                        column
                    )

            df = df[
                existing_columns
            ]

        except Exception as e:

            print(
                "Warning while checking existing CSV:",
                e
            )

    # Append
    df.to_csv(
        path,
        mode="a",
        header=not os.path.exists(path),
        index=False
    )

    print(
        f"✓ Appended {len(df)} rows → {path}"
    )

In [76]:
append_to_csv(
    candidate_df,
    "candidates.csv"
)

append_to_csv(
    pages_df,
    "pages.csv"
)

append_to_csv(
    links_df,
    "links.csv"
)

append_to_csv(
    skills_df,
    "skills_evidence.csv"
)

append_to_csv(
    projects_df,
    "projects.csv"
)

append_to_csv(
    experience_df,
    "experience.csv"
)

append_to_csv(
    education_df,
    "education.csv"
)

append_to_csv(
    certifications_df,
    "certifications.csv"
)

append_to_csv(
    documents_df,
    "documents.csv"
)

✓ Appended 1 rows → portfolio_data/candidates.csv
✓ Appended 1 rows → portfolio_data/pages.csv
✓ Appended 5 rows → portfolio_data/links.csv
✓ Appended 80 rows → portfolio_data/skills_evidence.csv
No records to save: projects.csv
No records to save: experience.csv
✓ Appended 10 rows → portfolio_data/education.csv
✓ Appended 5 rows → portfolio_data/certifications.csv
✓ Appended 1 rows → portfolio_data/documents.csv
